(sec:loc_prop)=
# Localized properties

(sec:esp)=
## ESP charges

Since there is no unique definition for partial charges and no corresponding physical observable, they can be assigned in several ways, such as being derived from the quantum mechanical electrostatic potential

\begin{equation*}
V(\mathbf{r}) = 
\sum_{I}
\frac{Z_I e}{4\pi\varepsilon_0 |\mathbf{r}-\mathbf{R}_I|} - e
\sum_{\mu,\nu}
D_{\mu\nu}
\int 
\frac{
\phi_\mu^*(\mathbf{r}')\phi_\nu(\mathbf{r}')
}{
4\pi\varepsilon_0
|\mathbf{r}-\mathbf{r}'|
}
d^3\mathbf{r}'
\end{equation*}

that can be replaced with a potential caused by the partial charges:

\begin{equation*}
\widetilde{V}(\mathbf{r}) = 
\sum_{I}
\frac{
q_I
}{
4\pi\varepsilon_0
|\mathbf{r}-\mathbf{R}_I|
}
\end{equation*}

The Merz–Kollman scheme minimizes the squared norm difference between these two quantities evaluated on a set of grid points in the solvent-accessible region of the molecule with respect to variations in the partial charges and a constraint of a conservation of the total molecular charge – the grid points are distributed on successive layers of scaled van der Waals surfaces. This measure is referred to as the figure-of-merit

\begin{equation*}
\chi_{\mathrm{esp}}^2 = \sum_a \bigl(V(\mathbf{r}_a) - \widetilde{V}(\mathbf{r}_a)\bigl)^2
\end{equation*}

The resulting electrostatic potential (ESP) charges are obtained by solving the equation

\begin{equation*}
\mathbf{A} \, \mathbf{q} = \mathbf{b}
\end{equation*}

where

\begin{equation*}
A_{IJ} =
\frac{1}{4\pi\varepsilon_0}
\sum_{a} \frac{1}{r_{aI} r_{aJ}}
\end{equation*}

and

\begin{equation*}
b_I = \sum_{a} 
\frac{
V(\mathbf{r}_a)
}{
r_{aI}
}
\end{equation*}

**Python script**

In [1]:
import veloxchem as vlx

xyz_str = """6
Methanol
H      1.2001      0.0363      0.8431
C      0.7031      0.0083     -0.1305
H      0.9877      0.8943     -0.7114
H      1.0155     -0.8918     -0.6742
O     -0.6582     -0.0067      0.1730
H     -1.1326     -0.0311     -0.6482
"""

molecule = vlx.Molecule.read_xyz_string(xyz_str)
basis = vlx.MolecularBasis.read(molecule, "6-31G*")

esp_drv = vlx.EspChargesDriver()
esp_drv.equal_charges = "1=3, 1=4"
esp_charges = esp_drv.compute(molecule, basis)

                                                                                                                          
                                            Self Consistent Field Driver Setup                                            
                                                                                                                          
                   Wave Function Model             : Spin-Restricted Hartree-Fock                                         
                   Initial Guess Model             : Superposition of Atomic Densities                                    
                   Convergence Accelerator         : Two Level Direct Inversion of Iterative Subspace                     
                   Max. Number of Iterations       : 50                                                                   
                   Max. Number of Error Vectors    : 10                                                                   
                

**Text file**

:::{code}
@jobs
task: esp charges
@end

@method settings
basis: 6-31G*
@end

@molecule
charge: 0
multiplicity: 1
xyz:  
...
@end
:::

In both cases, the user can control the number of layers of the molecular surface as well as the surface grid point density in these layers (in units of Å$^{-2}$). In the above examples, the recommended default values are employed.

(sec:resp)=
## RESP charges

The restrained electrostatic potential (RESP) charge model is an improvement to the Merz–Kollman scheme as the figure-of-merit $\chi^2_\mathrm{esp}$, is rather insensitive to variations in charges of atoms buried inside the molecule, as illustrated below for methanol and its buried carbon atom in red.

:::{image} ../images/chi_square.png
:name: chi_square
:width: 500px
:align: center
:::

To avoid unphysically large charges of interior atoms, a hyperbolic penalty function is added

\begin{equation*}
\chi_{\mathrm{resp}}^2 = \alpha \sum_I \bigl((q_I^2+\beta^2)^{1/2}-\beta\bigl)
\end{equation*}

so that the diagonal matrix elements become equal to

\begin{equation*}
A_{II} = 
\frac{1}{4\pi\varepsilon_0}
\sum_{a} \frac{1}{r_{aI}^2} + \alpha \, (q_I^2+\beta^2)^{-1/2}
\end{equation*}

with a dependency on the partial charge. Consequently, RESP charges are obtained by solving the matrix equation iteratively until the charges and Lagrange multipliers become self-consistent. In addition to that, the RESP charge model allows for the introduction of constraints on charges of equivalent atoms due to symmetry operations or bond rotations.

**Python script**

In [2]:
import veloxchem as vlx

xyz_str = """6
Methanol
H      1.2001      0.0363      0.8431
C      0.7031      0.0083     -0.1305
H      0.9877      0.8943     -0.7114
H      1.0155     -0.8918     -0.6742
O     -0.6582     -0.0067      0.1730
H     -1.1326     -0.0311     -0.6482
"""

molecule = vlx.Molecule.read_xyz_string(xyz_str)
basis = vlx.MolecularBasis.read(molecule, "6-31G*")

resp_drv = vlx.RespChargesDriver()
resp_drv.equal_charges = "1=3, 1=4"
resp_charges = resp_drv.compute(molecule, basis)

                                                                                                                          
                                            Self Consistent Field Driver Setup                                            
                                                                                                                          
                   Wave Function Model             : Spin-Restricted Hartree-Fock                                         
                   Initial Guess Model             : Superposition of Atomic Densities                                    
                   Convergence Accelerator         : Two Level Direct Inversion of Iterative Subspace                     
                   Max. Number of Iterations       : 50                                                                   
                   Max. Number of Error Vectors    : 10                                                                   
                

**Text file**

:::{code}
@jobs
task: resp charges
@end

@method settings
basis: 6-31g*
@end

@resp charges
equal charges: 2 = 3    ! with reference to the atom ordering below
@end

@molecule
charge: 0
multiplicity: 1
xyz:  
...
@end 
:::

(subsec:bol-weighted-resp)=
### Boltzmann-weighted RESP charges

It is also possible to calculate the Boltzmann-weighted RESP charges for a set of conformers.

**Python script**

Below, the argument `conformers` is a list of molecule objects for which the averaging is performed. By default, the calculation is performed at the Hartree–Fock level using the 6-31G* basis set. 

In [3]:
import veloxchem as vlx

molecule = vlx.Molecule.read_name("propanol")

confgen = vlx.ConformerGenerator()
conformers = confgen.generate(molecule)

resp_drv = vlx.RespChargesDriver()
resp_charges = resp_drv.compute(conformers["molecules"])

Reading propanol from PubChem...

Reference: S. Kim, J. Chen, T. Cheng, A. Gindulyte, J. He, S. He, Q. Li, B. A. Shoemaker, P. A. Thiessen, B. Yu, L. Zaslavsky, J. Zhang, E. E. Bolton, Nucleic Acids Res., 2025, 53, D1516-D1525.

Please double-check the compound since names may refer to more than one record.

                                                                                                                          
                                            Self Consistent Field Driver Setup                                            
                                                                                                                          
                   Wave Function Model             : Spin-Restricted Hartree-Fock                                         
                   Initial Guess Model             : Superposition of Atomic Densities                                    
                   Convergence Accelerator         : Two Level Direct Inver

In [36]:
print("Atom  Charge")
for idx, label in enumerate(molecule.get_labels()):
    print(f"{label}{idx + 1:<2} {resp_charges[idx]:7.2f}")

Atom  Charge
C1    -0.04
C2     0.06
C3     0.20
O4    -0.59
H5     0.01
H6     0.01
H7     0.01
H8    -0.01
H9    -0.01
H10    0.00
H11    0.00
H12    0.36


In [37]:
emin = conformers["energies"][0]

print(f"Energy: {conformers["energies"][0] - emin:4.2f} kJ/mol")
conformers["molecules"][0].show(atom_labels=True, atom_indices=True)

print(f"Energy: {conformers["energies"][1] - emin:4.2f} kJ/mol")
conformers["molecules"][1].show()

print(f"Energy: {conformers["energies"][2] - emin:4.2f} kJ/mol")
conformers["molecules"][2].show()

print(f"Energy: {conformers["energies"][3] - emin:4.2f} kJ/mol")
conformers["molecules"][3].show()

Energy: 0.00 kJ/mol


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

Energy: 0.11 kJ/mol


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

Energy: 4.36 kJ/mol


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

Energy: 4.36 kJ/mol


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

**Text file**

:::{code}
@jobs
task: resp charges
@end

@method settings
basis: 6-31g*
@end

@resp charges
xyz_file: all_conformers.xyz
@end

@molecule
charge: 0
multiplicity: 1
@end
:::

(sec:chelpg)=
## CHELPG charges

Different choices of grid points in the Merz–Kollman (MK) scheme can be made. CHELPG charges are obtained with grid points chosen on a dense cubic grid with exclusion made of grid points inside the van der Waals molecular volume.

In contrast to the original MK scheme, the calculation of CHELPG charges involve grid points directly outside the van der Waals molecular volume, and since the electrostatic potential is here large, these points will be important for the minimization of the Lagrangian. We note that there is no universal grid-point choice that can be considered best for all situations.

**Python script**

In [12]:
import veloxchem as vlx

xyz_str = """6
Methanol
H      1.2001      0.0363      0.8431
C      0.7031      0.0083     -0.1305
H      0.9877      0.8943     -0.7114
H      1.0155     -0.8918     -0.6742
O     -0.6582     -0.0067      0.1730
H     -1.1326     -0.0311     -0.6482
"""

molecule = vlx.Molecule.read_xyz_string(xyz_str)
basis = vlx.MolecularBasis.read(molecule, "6-31G*")

esp_drv = vlx.EspChargesDriver()

esp_drv.grid_type = "chelpg"

esp_drv.equal_charges = "1=3, 1=4"

chelpg_charges = esp_drv.compute(molecule, basis)

                                                                                                                          
                                            Self Consistent Field Driver Setup                                            
                                                                                                                          
                   Wave Function Model             : Spin-Restricted Hartree-Fock                                         
                   Initial Guess Model             : Superposition of Atomic Densities                                    
                   Convergence Accelerator         : Two Level Direct Inversion of Iterative Subspace                     
                   Max. Number of Iterations       : 50                                                                   
                   Max. Number of Error Vectors    : 10                                                                   
                

## Charge comparison

The localized charges for methanol in the examples above become:

In [13]:
print("Atom      ESP charge        RESP charge    CHELPG charge")

print(56 * "-")

for label, esp_charge, resp_charge, chelpg_charge in zip(
    molecule.get_labels(), esp_charges, resp_charges, chelpg_charges
):

    print(
        f"{label :s} {esp_charge : 18.6f}{resp_charge : 18.6f}{chelpg_charge : 18.6f}"
    )

print(56 * "-")

print(
    f"Total: {esp_charges.sum() : 13.6f}{resp_charges.sum() : 18.6f}{chelpg_charges.sum() : 18.6f}"
)

Atom      ESP charge        RESP charge    CHELPG charge
--------------------------------------------------------
H           0.023220          0.033747          0.003200
C           0.148458          0.118610          0.225480
H           0.023220          0.033747          0.003200
H           0.023220          0.033747          0.003200
O          -0.594785         -0.639004         -0.611345
H           0.376669          0.419154          0.376265
--------------------------------------------------------
Total:     -0.000000          0.000000          0.000000


(sec:loprop)=
## LoProp charges and polarizabilities

The LoProp approach {cite}`Gagliardi2004` is implemented for the determination of localized (atomic) charges and polarizabilities that enter into polarizable embedding calculations of optical spectra.

**Python script**

In [14]:
import veloxchem as vlx

molecule = vlx.Molecule.read_molecule_string("""
O    0.0000000    0.0000000   -0.1653507
H    0.7493682    0.0000000    0.4424329
H   -0.7493682    0.0000000    0.4424329
""")

basis = vlx.MolecularBasis.read(molecule, "ANO-S-VDZP")

scf_drv = vlx.ScfRestrictedDriver()
scf_results = scf_drv.compute(molecule, basis)

loprop_drv = vlx.PEForceFieldGenerator()
loprop_results = loprop_drv.compute(molecule, basis, scf_results)

                                                                                                                          
                                            Self Consistent Field Driver Setup                                            
                                                                                                                          
                   Wave Function Model             : Spin-Restricted Hartree-Fock                                         
                   Initial Guess Model             : Superposition of Atomic Densities                                    
                   Convergence Accelerator         : Two Level Direct Inversion of Iterative Subspace                     
                   Max. Number of Iterations       : 50                                                                   
                   Max. Number of Error Vectors    : 10                                                                   
                

This calculation gives the following results.

In [15]:
print("LoProp charges (a.u.):")
print(f"O: {loprop_results['localized_charges'][0] : .4f}")
print(f"H: {loprop_results['localized_charges'][1] : .4f}")
print(f"H: {loprop_results['localized_charges'][2] : .4f}")

print("\nLoProp polarizabilities (a.u.):")
print("     xx     yy     zz")
print(
    f"O: {loprop_results['localized_polarizabilities'][0][0]:5.2f}{loprop_results['localized_polarizabilities'][0][3]:7.2f}{loprop_results['localized_polarizabilities'][0][5]:7.2f}"
)
print(
    f"H: {loprop_results['localized_polarizabilities'][1][0]:5.2f}{loprop_results['localized_polarizabilities'][1][3]:7.2f}{loprop_results['localized_polarizabilities'][1][5]:7.2f}"
)
print(
    f"H: {loprop_results['localized_polarizabilities'][2][0]:5.2f}{loprop_results['localized_polarizabilities'][2][3]:7.2f}{loprop_results['localized_polarizabilities'][2][5]:7.2f}"
)

LoProp charges (a.u.):
O: -0.6777
H:  0.3388
H:  0.3388

LoProp polarizabilities (a.u.):
     xx     yy     zz
O:  3.88   2.91   3.69
H:  1.84   1.21   1.57
H:  1.84   1.21   1.57


:::{image} ../images/water.png
:align: center
:width: 200px
:::

**Text file**
:::{code}
@jobs
task: loprop
@end

@method settings
xcfun: b3lyp
basis: ANO-S-VDZP ! An ANO type of basis set should be used
@end

@molecule
charge: 0
multiplicity: 1
xyz:
...
@end
:::